In [ ]:
# Importing dependency
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras import datasets
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
import numpy as np
import tensorflow as tf

tf.keras.backend.clear_session()  # For easy reset of notebook state.
tf.compat.v1.disable_eager_execution()

# Import Data
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Build Model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(10))

# Model Summary
model.summary()

# Model Compile 
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Define the Gradient Fucntion
epoch_gradient = []

wnames = tf.nest.map_structure(lambda w: w.name, model.trainable_weights)

# Define the Gradient Function
def get_gradient_func(model):
    grads = K.gradients(model.total_loss, model.trainable_weights)
    inputs = model._feed_inputs + model._feed_targets + model._feed_sample_weights
    func = K.function(inputs, grads)
    return func


  # def compute_gradients(self):
  #   x, y, sample_weight = data_adapter.unpack_x_y_sample_weight(self.dataset)
  #   # Run forward pass
  #   with tf.GradientTape() as tape:
  #     y_pred = self.model(x, training=True)
  #     loss = self.model.compute_loss(x, y, y_pred, sample_weight)
  #   self.model._validate_target_and_loss(y, loss)
  #   # Run backwards pass
  #   return self.model.optimizer.compute_gradients(
  #     loss=loss, var_list=self.trainable_variables, tape=tape
  #   )



# Define the Required Callback Function
class GradientCalcCallback(keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
      get_gradient = get_gradient_func(model)
      grads = get_gradient([train_images, train_labels, None])
      grads = dict(zip(wnames, grads))
      epoch_gradient.append(grads)

epoch = 2

model.fit(train_images, train_labels, epochs=epoch, validation_data=(test_images, test_labels), callbacks=[GradientCalcCallback()])


# (7) Convert to a 2 dimensiaonal array of (epoch, gradients) type
print("Total number of epochs run:", epoch)
print("Gradient Array has the shape:",epoch_gradient)

In [ ]:
import h5py
import numpy as np

In [ ]:
data_dict = {
    '0': {
        'a_array': np.random.random(size = 4),
        'b_array': 10 * np.random.random(size = 4),
        'c_array': 100 * np.random.random(size = 4),
        'd_scalar': 1.23
        },
    '1': {
        'a_array': np.random.random(size = 4),
        'b_array': 10 * np.random.random(size = 4),
        'c_array': 100 * np.random.random(size = 4),
        'd_scalar': 12.3
        }
}